### setup

In [1]:
# !pip install -U sentence-transformers
# !pip install datasets
# !pip install tensorflow_ranking

In [2]:
import pickle
import numpy as np
import os
import tqdm
import torch
import gc

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

from importlib import reload
from matplotlib import pyplot as plt

import matrix_approx_zeshel

matrix_approx_zeshel = reload(matrix_approx_zeshel)

!ls  | grep matrix_approx_zeshel

2024-03-27 04:50:11.736034: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-03-27 04:50:11.853243: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-03-27 04:50:11.854400: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-03-27 04:50:13.740983: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


matrix_approx_zeshel.py


In [3]:
import collections
import pickle
import numpy as np
import tqdm
import os
import gc

import matplotlib.pyplot as plt


def load(limit, raw_path = "stand/log.local.logtime2.txt", path = "log.local.logtime2.bin", key_games = None, seed = 17, det_attempts = 0, key_size = 100):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            flimit, frequests, fdocembs = pickle.load(f)
            if flimit == limit:
                requests, docembs = frequests, fdocembs
            else:
                print(f"WARN: buffered limit is different, {flimit} != {limit}, reloading...")

    if not requests:
        with open(raw_path) as f:
            req = list()
            reqid = None
            models = list()
            prevreqmodel = None
            reqmodel = dict()
            prevmodelid = -1
            bannermodelid = -1
            for i, line in tqdm.tqdm_notebook(enumerate(f)):
                if line.startswith("Model = 6;"):
                    prevreqmodel = reqmodel
                    reqmodel = dict()

                if line.startswith("Model = "):
                    spl = line.split(" ")
                    prevmodelid = int(spl[2][:-1])
                    bannermodelid = max(bannermodelid , prevmodelid)
                    reqmodel[prevmodelid] = readvector(spl[3])
                elif line.startswith("dbid"):
                    spl = line.split(" ")
                    dbid = int(spl[1][:-1])
                    docembs[bannermodelid][dbid] = readvector(spl[2])
                elif line.startswith("seed"):
                    if len(requests) >= limit:
                        break
                    if req:
                        requests.append((reqid, prevreqmodel, sorted(req)))
                        req = list()
                    reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
                else:
                    req.append(
                        (int(line.split()[0]), float(line.split()[1]))
                    )
        
        with open(path, "wb") as f:
            pickle.dump((limit, requests, docembs), f)

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = key_size, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
            self.games_count = games_count
            
            self.key_size = key_size
            self.key_games = (
                np.random.choice(np.arange(games_count), key_size, replace=False)
                if key_games is None else
                key_games
            )

            self.requests = requests
            np.random.seed(seed)
            np.random.shuffle(self.requests)
            
            self.try_det_attempts(det_attempts)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks
            self.relevs = dict()
            
        def get_top_games(self):
            if not hasattr(self, "top_games"):
                embed_games = np.array([
                    np.array([r[2][g_i][1] for r in self.get_requests("train")])
                    for g_i in range(self.games_count)
                ])

                self.embed_games_mean = embed_games.mean(axis=1)
                self.top_games_all = (-self.embed_games_mean).argsort()
                self.top_games = self.top_games_all[:len(self.key_games)]

            return self.top_games
        
        def set_top_games_as_key(self):
            self.key_games = self.get_top_games()
            return self
        
        def get_kmeans_games(self, all_from_labels=True):
            X = self.get_relevs("train").T

            k_func = (
                (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
                if not all_from_labels else
                (lambda C : from_labels(X, C.labels_))
            )
            K_KMeans = k_func(
                KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
            )

            return K_KMeans

        def set_kmeans_games_as_key(self, *args, **kwargs):
            self.key_games = self.get_kmeans_games(*args, **kwargs)
            return self

        def try_det_attempts(self, det_attempts, model_id = 6):
            def get_det(r, r_i_array):
                kr = np.array([
                    r[r_i][1][model_id][:self.key_size]
                    for r_i in r_i_array[:self.key_size]
                ])
                return np.abs(np.linalg.det(kr[:kr.shape[1], :]))

            best_i_array = np.arange(len(self.requests))

            for _ in range(det_attempts):
                # print("try update key_reqs...")
                
                r_i_array = np.arange(len(self.requests))
                np.random.shuffle(r_i_array)
                
                n, o = get_det(self.requests, r_i_array), get_det(self.requests, best_i_array)
                # print(n, o)
                if n > o:
                    best_i_array = r_i_array
                    # print("updated!")

            print("Best det = ", get_det(self.requests, best_i_array))
            
            new_requests = [
                self.requests[i]
                for i in best_i_array
            ]
            
            del self.requests
            gc.collect()

            self.requests = new_requests
            print(get_det(self.requests, np.arange(len(self.requests))))

        def get_relevs(self, t = "train"):
            if t not in self.relevs:
                self.relevs[t] = np.array([
                    np.array([g_i[1] for g_i in r[2]])
                    for r in self.get_requests(t)
                ])
                
            return self.relevs[t]

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext(key_games = key_games, seed = seed, det_attempts = det_attempts)

In [4]:
def load_ment_to_ent_scores(directory = "yugioh", shuffle_rows = 0, full = True):
    data = list()

    for file in os.listdir(directory):
        path = f"{directory}/{file}"
        print(f"Loading file {path}")
        with open(path, "rb") as f:
            data.append(
                pickle.load(f)
            )
    data = sorted(data, key = lambda x: x["arg_dict"]["n_ment_start"])

    for i in range(len(data) - 1):
        if full:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] == data[i + 1]["arg_dict"]["n_ment_start"]
        else:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] <= data[i + 1]["arg_dict"]["n_ment_start"]
        
    ment_to_ent_scores = list(map(lambda x: x["ment_to_ent_scores"], data))
    ment_to_ent_scores = np.vstack(ment_to_ent_scores)
    print("Loaded shape = ", ment_to_ent_scores.shape)
    
    if shuffle_rows:
        print(f"Shuffling... (seed = {shuffle_rows})")
        np.random.seed(shuffle_rows)
        np.random.shuffle(ment_to_ent_scores)
    
    return ment_to_ent_scores

In [5]:
from sklearn.cluster import KMeans
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances


def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

class EvalContextRelevs:
    def __init__(self, relevs, key_size = 100, train_size = 0.7, key_games = None, seed = 17, shuffle=False, det_attempts = 0):
        self.relevs = np.array(relevs)
        self.reqs_count = self.relevs.shape[0]
        self.games_count = self.relevs.shape[1]
        
        self.key_size = key_size
        self.key_games = (
            np.random.choice(np.arange(self.games_count), key_size, replace=False)
            if key_games is None else
            key_games
        )
        np.random.seed(seed)
        if shuffle:
            np.random.shuffle(self.relevs)

        self.try_det_attempts(det_attempts)
        self.train_split = int(self.reqs_count * train_size)

        assert key_size + 1 < self.train_split


        self.key_relevs = self.relevs[:key_size]
        self.train_relevs = self.relevs[key_size + 1: self.train_split]
        self.test_relevs = self.relevs[self.train_split:]

        self.slices = ["key", "train", "test"]
        print(len(self.key_relevs), len(self.train_relevs), len(self.test_relevs))

    def get_top_games(self):
        return self.relevs.mean(axis=0).argsort()[:self.key_size]

    def set_top_games_as_key(self):
        self.key_games = self.get_top_games()
        return self

    def get_kmeans_games(self, all_from_labels=True):
        X = self.get_relevs("train").T

        k_func = (
            (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
            if not all_from_labels else
            (lambda C : from_labels(X, C.labels_))
        )
        K_KMeans = k_func(
            KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
        )
        
        return K_KMeans
    
    def set_kmeans_games_as_key(self, *args, **kwargs):
        self.key_games = self.get_kmeans_games(*args, **kwargs)
        return self
    

    def try_det_attempts(self, det_attempts):
        def get_det(r):
            kr = r[:self.key_size, self.key_games] - r.mean()
            return np.abs(np.linalg.det(kr))

        best_i_array = np.arange(len(self.relevs))

        for i in range(det_attempts):

            r_i_array = np.arange(len(self.relevs))
            np.random.shuffle(r_i_array)

            n, o = get_det(self.relevs[r_i_array, :]), get_det(self.relevs[best_i_array, :])
            
            # print(f"try update key_reqs ({o} vs {n}...")
            if n > o:
                best_i_array = r_i_array
                print(f"updated det ({i}, {o} -> {n})")

        print("Best det = ", get_det(self.relevs[best_i_array, :]))

        self.relevs = self.relevs[best_i_array, :]
        print("Current de = ", get_det(self.relevs))
        
    def get_relevs(self, t = "train"):
        if t == "train":
            return self.train_relevs
        elif t == "key":
            return self.key_relevs
        elif t == "test":
            return self.test_relevs
        else:
            assert False
            
    def get_requests(self, t = "train"):
        if t == "train":
            return self.train_reqs
        elif t == "key":
            return self.key_reqs
        elif t == "test":
            return self.test_reqs
        else:
            assert False

In [6]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        self.trus_top = dict()
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):    
        recs = self.recommend(t)
        
        if isinstance(recs, list):
            recs = np.array(recs)
        
        mse = np.mean((recs - self.get_user_scores(t)) ** 2)
        
        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :topsize].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :topsize]
        
        if (t, topsize) not in self.trus_top:
            trus = self.get_user_scores(t)
            trus = np.argsort(-trus, axis=1)[:, :topsize]
            self.trus_top[(t, topsize)] = trus
            
        trus = self.trus_top[(t, topsize)]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)
    
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "matrix_l2": 0,
    "dssm_l2": 0,
    "train_popbias": False,
    "train_bias": False,
    "train_vbias": False,
    "verbose": True,
    "train_matrix": True,
    "train_dssm": False,
    "loss": "mse",
    "ubatch": 1e9,
    "activation": "relu",
    "score_verbose": 0,
    "trainable_items": False,
    "use_keys_in_train": False,
    "Winit": "glorot0.01",
    "Wfreeze": False,
    "TEinit": "legacy",
    "normalize_embs": True,
    "save_callback": None,
    "inner_dim": None
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(fit_kwargs)
        self.fit_kwargs = p
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[i] for i in sorted(ctx.key_games)])
                    for r_i in ctx.get_relevs(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[g_i] for r in ctx.get_relevs("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        
        if self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["TEinit"] == "legacy":
                value = self.embed_games
            elif self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                value = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print(type(value))
            else:
                assert False, "unknown TEinit: " + str(self.fit_kwargs["TEinit"])
                
            self.trainable_games = tf.Variable(
                tf.convert_to_tensor(value, dtype=tf.float32),
                trainable=True
            )
        else:
            if self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                self.embed_games = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print("ANNCur init")
            
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = ctx.get_relevs("key")
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
        
        self.slice2best = {
            t : 0
            for t in ctx.slices
        }
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        if self.fit_kwargs["normalize_embs"]:
            return (self.embed_users[t] - self.embed_users_mean["train"]) / self.embed_users[t].std(axis=0)
        else:
            return self.embed_users[t]
    
    def get_game_embs(self):
        if not self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["normalize_embs"]:
                return (self.embed_games - self.embed_games.mean(axis=0)) / self.embed_games.std(axis=0)
            else:
                return self.embed_games 
        else:
            return self.trainable_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.p = p
        self.train_bias = p["train_bias"]
        self.train_vbias = p["train_vbias"]
        self.train_popbias = p["train_popbias"]
        self.train_matrix = p["train_matrix"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        if p["use_keys_in_train"]:
            train_user_scores = np.vstack([
                self.get_user_scores("key"),
                self.get_user_scores("train")
            ])
            train_user_embs = np.vstack([
                self.get_user_embs("key"),
                self.get_user_embs("train")
            ])
        else:
            train_user_scores = self.get_user_scores("train")
            train_user_embs = self.get_user_embs("train")

        game_embs = self.get_game_embs()
        
        
        
        if self.p["Winit"] == "glorot0.01":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
            values = values / 100.  
        elif self.p["Winit"] == "glorot":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        elif self.p["Winit"] == "eye":
            values = np.eye(train_user_embs.shape[1], game_embs.shape[1])
        else:
            assert False, "init is not known:" + str(self.p["init"])
            
        self.W = tf.Variable(values, trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            inner_dim = p["inner_dim"] if p.get("inner_dim", None) is not None else dim
            print(f"dssm_dim = {dim}, inner_dim={inner_dim}")
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(inner_dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(inner_dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        if self.train_vbias:
            self.vb = tf.Variable(
                np.zeros_like(self.game_avg_scores["train"], dtype=np.float32),
                trainable = True
            ) 
        
        
        for i in range(p["n"]):
            def loss():
                def get_logits_scores(loss_batch = 1e9):
                    game_slice = None
                    
                    ubatch = p["ubatch"]
                    if ubatch >= train_user_scores.shape[0]:
                        train_user_embs_ = train_user_embs
                        scores_ = train_user_scores
                    else:
                        u_slice = np.random.choice(np.arange(train_user_scores.shape[0]), ubatch, replace = True)
                        train_user_embs_ = train_user_embs[u_slice]
                        scores_ = train_user_scores[u_slice]
                    
                    if loss_batch >= game_embs.shape[0]:
                        game_embs_ = game_embs
                        scores_ = scores_
                    else:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        game_embs_ = (
                            game_embs[game_slice]
                            if isinstance(game_embs, np.ndarray) else
                            tf.gather(game_embs, game_slice)
                        )
                        scores_ = scores_[:, game_slice]
                        
                    
                    logits = 0
                    
                    if self.train_matrix:
                        logits += train_user_embs_ @ self.W @ tf.transpose(game_embs_)

                    if self.train_dssm:
                        logits += self.u_dssm(train_user_embs_) @ tf.transpose(self.g_dssm(game_embs_))
                    
                    if self.train_popbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.pb * self.game_avg_scores["train"]
                        else:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_vbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.vb
                        else:
                            logits += tf.gather(self.vb, game_slice)
                        
                    if self.train_bias:
                        logits += self.b
                        
                    return logits, scores_
                        
                if self.loss == "mse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    v = tf.reduce_mean((scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    q_mean = scores.mean(axis=1)
                    v = tf.reduce_mean(((scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    while True:
                        logits, scores = get_logits_scores(
                            loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            scores.astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                elif self.loss == "softmax_signed":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    sf = tf.nn.softmax(logits, axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        sf,
                        -sf
                    ))
                elif self.loss == "softmax_weighted":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ) * scores)
                elif self.loss == "sigmoid_top_100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    mask = np.argsort(-scores, axis=1) < 100
                    v = -tf.reduce_sum(
                        tf.math.log_sigmoid((2 * mask - 1) * logits)
                    )
                elif self.loss == "KL100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    mask = np.argsort(-scores, axis=1) < 100
                    v = tf.keras.losses.KLDivergence()((scores.T >= qs.T).T, logits)
                else:
                    assert False
                    
                if self.p["c"]:
                    v += tf.reduce_mean(self.W * self.W) * p["c"]
                    
                if self.p["dssm_l2"]:
                    for weights_ in [self.u_dssm.weights, self.g_dssm.weights]:
                        for weight_ in weights_:
                            v += tf.reduce_sum(weight_ * weight_) * self.p["dssm_l2"]
                
                if self.p["matrix_l2"]:
                    v += tf.reduce_sum(self.W * self.W) * self.p["matrix_l2"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            
            if self.train_matrix and (not self.p["Wfreeze"]):
                weights += [self.W]

            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_vbias:
                weights += [self.vb]   
                
            if self.train_bias:
                weights += [self.b]   
                
            
            if self.p["trainable_items"]:
                weights += [self.trainable_games]
                
                
            opt.minimize(loss, weights)
            
            if p["score_verbose"] and (i % p["score_verbose"] == 0):
                print(f"\n=== Iteration {i} ===")
                score = dict()
                for sl in self.ctx.slices:
                    score[sl] = self.get_score(sl)
                    print(f"slice = {sl}, score = {score[sl]}")
                    
                if score["train"] > self.slice2best["train"]:
                    self.slice2best = score
                    if p["save_callback"] is not None:
                        p["save_callback"](self)
                
        print("last loss = ", loss())

    def recommend(self, t):
        logits = 0
                    
        if self.train_matrix:
            logits += self.get_user_embs(t) @ self.W @ tf.transpose(self.get_game_embs())

        if self.train_dssm:
            logits += self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_vbias:
            logits += self.vb
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))
            
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))
            
class AnnCUR(Popular):
    def __init__(self, ctx, oracle=False, key_games=None, name=None):
        super().__init__(ctx)
        self.oracle = oracle
        self.ctx = ctx
        self.name = name

        self.key_cols_idx = np.array(sorted(ctx.key_games if key_games is None else key_games))
        rows_idx = np.arange(ctx.get_relevs("train").shape[0])
        
        self.cur = matrix_approx_zeshel.CURApprox(
            rows=torch.from_numpy(ctx.get_relevs("train")),
            cols=torch.from_numpy(ctx.get_relevs("train")[:, self.key_cols_idx]),
            row_idxs=rows_idx,
            col_idxs=self.key_cols_idx,
            approx_preference="rows",
            A=(torch.from_numpy(ctx.get_relevs("train")) if oracle else None)
        )        
        
    def __repr__(self):
        if self.name is None:
            return f"AnnCur({id(self)})"
        else:
            return f"AnnCur({self.name} - {id(self)})"
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        return self.cur.latent_cols.T

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        key_relevs = torch.from_numpy(self.ctx.get_relevs(t)[:, self.key_cols_idx])
        return self.cur.get_complete_row(key_relevs)
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

In [7]:
def coitem_algorithm(n_support, candidate_items, target_items, min_item_rel_norm=1e-5, eps=1e-6):
    support_items = []
    support_items_scores = []
    n_c, n_q = candidate_items.shape
    n_t = target_items.shape[0]

    candidate_item_squared_norms = (candidate_items ** 2).sum(axis=1)
    min_item_norm = min_item_rel_norm * np.sqrt(candidate_item_squared_norms.mean())
    
    candidate_coitems = candidate_items.dot(
        target_items.T.dot(target_items)
    )
    orthonormed_support_items = np.zeros((n_support, n_q))
    remaining_items = np.ones(candidate_items.shape[0], dtype="bool")
    for t in tqdm.tqdm_notebook(range(n_support)):
        scores = (candidate_coitems * candidate_items).sum(axis=1)
        remaining_items &= (candidate_item_squared_norms >= min_item_norm ** 2)
        scores[remaining_items] /= candidate_item_squared_norms[remaining_items]
        scores[~remaining_items] = 0
        
        new_item_id = np.argmax(scores)
        assert remaining_items[new_item_id]
        support_items.append(new_item_id)
        support_items_scores.append(scores[new_item_id] / (n_t * n_q))
        remaining_items[new_item_id] = False
        
        new_item = candidate_items[new_item_id].copy()
        new_item -= orthonormed_support_items[:t].T.dot(
            orthonormed_support_items[:t].dot(new_item)
        )
        assert np.allclose((new_item ** 2).sum(), candidate_item_squared_norms[new_item_id], atol=eps)
        new_item /= np.sqrt((new_item ** 2).sum())
        orthonormed_support_items[t] = new_item
        new_coitem = candidate_coitems[new_item_id] / np.sqrt(candidate_item_squared_norms[new_item_id])
        
        coefs = (candidate_items * new_item).sum(axis=1)
        candidate_item_squared_norms -= coefs ** 2
        assert np.all(candidate_item_squared_norms[remaining_items] > -eps)
        
        candidate_coitems -= coefs.reshape((-1, 1)).dot(new_coitem.reshape((1, -1)))
        cocoefs = (candidate_coitems * new_item).sum(axis=1, keepdims=True)
        candidate_coitems -= cocoefs.dot(new_item.reshape((1, -1)))
    return np.array(support_items, dtype=np.int64), np.array(support_items_scores)

# Preparate

In [8]:
def do(ctx, name, coitem=True, kmeans=True, top=True, random=True):
    if coitem:
        t = ctx.get_relevs("train").T
        t = (t - t.mean()) / t.std()  # like in other comparisons with RBE
        ctx.key_games = list(coitem_algorithm(ctx.key_size, t, t, 1e-8, eps=1e9)[0])

        ma = AnnCUR(ctx, key_games=ctx.key_games)

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")

        GE = ma.get_game_embs()
        QE = torch.from_numpy(ma.ctx.get_relevs("test")[:, ma.key_cols_idx])

        fname = f"./GE_QE_AnnCURxCoItem_{name}_{str(round(te, 4))[2:]}.npz"
        np.savez_compressed(fname, QE=QE.numpy(), GE=GE.numpy())
        print(fname)
        
    if kmeans:
        ctx.set_kmeans_games_as_key()
        ma = AnnCUR(ctx, key_games=ctx.key_games)

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")

        GE = ma.get_game_embs()
        QE = torch.from_numpy(ma.ctx.get_relevs("test")[:, ma.key_cols_idx])

        fname = f"./GE_QE_AnnCURxKMeans_{name}_{str(round(te, 4))[2:]}.npz"
        np.savez_compressed(fname, QE=QE.numpy(), GE=GE.numpy())
        print(fname)
        
    if top:
        ctx.set_top_games_as_key()
        ma = AnnCUR(ctx, key_games=ctx.key_games)

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")

        GE = ma.get_game_embs()
        QE = torch.from_numpy(ma.ctx.get_relevs("test")[:, ma.key_cols_idx])

        fname = f"./GE_QE_AnnCURxTop_{name}_{str(round(te, 4))[2:]}.npz"
        np.savez_compressed(fname, QE=QE.numpy(), GE=GE.numpy())
        print(fname)
        
    if random:
        ctx.key_games = np.random.choice(np.arange(ctx.games_count), ctx.key_size, replace=False)
        ma = AnnCUR(ctx, key_games=ctx.key_games)

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")

        GE = ma.get_game_embs()
        QE = torch.from_numpy(ma.ctx.get_relevs("test")[:, ma.key_cols_idx])

        fname = f"./GE_QE_AnnCURxRandom_{name}_{str(round(te, 4))[2:]}.npz"
        np.savez_compressed(fname, QE=QE.numpy(), GE=GE.numpy())
        print(fname)

In [9]:
m_ = None
def do_RBE(ctx, name, coitem=True, kmeans=True, top=True, random=True, N = 100000, LR = 0.0005, train_matrix=True, inner_dim=None):
    def get_save_callback(t, fname):
        def save_callback(m):
            global m_
            m_ = m
            GE_ = m.get_game_embs()
            GE = np.hstack([
                GE_,
                m.g_dssm(GE_),
                m.vb.numpy().reshape((-1, 1)) ,  # Vertical bias
                ctx.get_relevs("train").mean(axis=0).reshape((-1, 1))  # popbias
            ])

            R_All = ctx.get_relevs(t)
            QE_ = m.get_user_embs(t)
            QE = np.hstack([
                QE_ @ m.W,
                m.u_dssm(QE_),
                np.ones((R_All.shape[0], 1)),
                np.ones((R_All.shape[0], 1)) * m.pb
            ])
            
            l, r = fname.split(".npz")
            
            train_score = m.get_score("train")
            score = m.get_score(t)
            fname_ = l + f"_{str(round(train_score, 4))}_{str(round(score, 4))}.npz" + r
            print(f"Saving {fname_} ...")
            np.savez_compressed(fname_, QE=QE, GE=GE)
        return save_callback
            
    if coitem:
        t = ctx.get_relevs("train").T
        t = (t - t.mean()) / t.std()  # like in other comparisons with RBE
        ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': train_matrix,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "inner_dim": inner_dim,
            "save_callback": get_save_callback("test", f"./GE_QE_RBExCoItem_{name}.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)
        
    if kmeans:
        ctx.set_kmeans_games_as_key()
        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': train_matrix,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "inner_dim": inner_dim,
            "save_callback": get_save_callback("test", f"./GE_QE_RBExKMeans_{name}.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)
        
    if top:
        ctx.set_top_games_as_key()
        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': train_matrix,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "inner_dim": inner_dim,
            "save_callback": get_save_callback("test", f"./GE_QE_RBExTop_{name}.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)
        
    if random:
        ma = CBKnnV0(ctx, fit_kwargs={
            'c': 0,
            'train_matrix': train_matrix,
            'train_dssm': True,
            'train_vbias': True,
            'train_popbias': True, 'train_bias': True,
            'verbose': False, 'loss': 'softmax_signed',
            'loss_batch': 128, 'loss_q': 0.99,
            # 'dssm_l2': 5e-5,
            'activation': 'elu',
            'n': N,
            # 'ubatch': 1500,
            'score_verbose': 1000,
            'trainable_items': False,
            "TEinit": "anncur",
            "use_keys_in_train": True, # <<< DIFF HERE
            "learning_rate": LR,
            "inner_dim": inner_dim,
            "save_callback": get_save_callback("test", f"./GE_QE_{name}_RBExRandom.npz")
        })

        ma.fit()
        tr, te = ma.get_score("train"), ma.get_score("test")
        print(tr, te)

In [10]:
def get_Rp_basic(GE, A, R, QE_i, L):
    Ae = GE[A]

    u = np.linalg.pinv(Ae.T @ Ae) @ Ae.T @ R
    assert u.shape == QE_i.shape
    Rp = GE @ (u * L + QE_i * (1 - L))
    
    return Rp


def do_AXN(ctx, loaded, get_Rp, Z="test", STRIP = 0.05, Ls=np.linspace(0, 1, 11), Ks=[25], T_TS_s = [(200, 100)], not_deduplicate = False):
    # q = ctx.get_requests(Z)
    R_All = ctx.get_relevs(Z)

    GE, QE = loaded["GE"], loaded["QE"]

    best = 0
    best_arg = None

    for randomFirst in [False]:
        for K in Ks:
            for L in Ls:
                for T, TS in T_TS_s:
                    results = list()


                    As = list()
                    for i in tqdm.tqdm(range(int(STRIP * R_All.shape[0]))):
                        A = (
                            ctx.key_games[:K]
                            if randomFirst else
                            (GE @ QE[i]).argsort()[::-1][:K]  # try best by DE
                        )
                        Rt = R_All[i]
                        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                        for _ in range(T // K - 1):
                            Rp = get_Rp(GE, A, Rt[A], QE[i], L)

                            An = list(A)
                            for ai in Rp.argsort()[::-1]:
                                if not_deduplicate or (ai not in An):
                                    An.append(ai)
                                if len(An) == len(A) + K:
                                    break
                            A = np.array(An)

                        assert len(A) == T
                        As.append(A)


                        A = sorted([
                            (-Rt[ai], ai)
                            for ai in A
                        ])[:TS]
                        A = [ai for _, ai in A]

                        pred_top, gt_top = A, Rt.argsort()[::-1][:TS]
                        results.append(ev(pred_top, gt_top) / float(TS))

                    if np.mean(results) > best:
                        best = np.mean(results)
                        best_arg = f"K = {K}, L = {L}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)}"
                        best_arg_d = {
                            "K": K,
                            "L": L,
                            "randomFirst": randomFirst,
                            "score": np.mean(results)
                        }
                    print(f"K = {K}, L = {L}, TS = {TS}, rndFirst = {randomFirst} np.mean(results) = {np.mean(results)} best = {best}")

    return best_arg

## RecSys LT

In [ ]:
L = 7000
N = 1000
DA = 50

In [ ]:
ctx = load(L, raw_path = "stand/log.local.logtime2.txt",
           path="log.local.logtime2.true.bin", seed=17, det_attempts=DA, key_size=100)

In [ ]:
do_RBE(ctx, "RecSysLT_noW_", N=300000, kmeans=False, top=False, random=False, train_matrix=False)

## RecSys PairWise

In [ ]:
del ctx
gc.collect()

In [ ]:
L = 7000
N = 1000
DA = 50


ctx = load(L, raw_path = "//dev/null/stand/log.local.txt",
           path="log.local.bin.old", seed=17, det_attempts=DA, key_size=100)
print("LOADED")

In [ ]:
do_RBE(ctx, "RecSysLT_noW_", N=300000, kmeans=False, top=False, random=False, train_matrix=False)

# ZESHEL

In [11]:
DATASETS = ["yugioh", "pro_wrestling", "star_trek", "doctor_who", "military"]

In [14]:
del ctx

NameError: name 'ctx' is not defined

In [15]:
gc.collect()

482

In [ ]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100,
        key_size=100
    )

    print("LOADED")
    
    do_RBE(ctx, "RecSysLT_noW_", N=200000, kmeans=False, top=False, random=False, inner_dim=200)
    
    del ctx
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
LOADED


/var/tmp/ipykernel_3818262/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 10031)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (2260, 100)
dssm_dim = 100, inner_dim=200

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.040299999999999996 tf.Tensor(37.455357, shape=(), dtype=float32) 100
slice = key, score = 0.040299999999999996
np.mean(results), mse, len(results) =  0.0444424778761062 tf.Tensor(38.462288, shape=(), dtype=float32) 2260
slice = train, score = 0.0444424778761062
np.mean(results), mse, len(results) =  0.04584402764067127 tf.Tensor(38.723866, shape=(), dtype=float32) 1013
slice = test, score = 0.04584402764067127
np.mean(results), mse, len(results) =  0.0444424778761062 tf.Tensor(38.462288, shape=(), dtype=float32) 2260
np.mean(results), mse, len(results) =  0.04584402764067127 tf.Tensor(38.723866, shape=(), dtype=float32) 1013
Saving ./GE_QE_RBEx

In [14]:
for dataset in DATASETS:
    
    print ("\n\n\nDATASET = ", dataset)
    
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=dataset),
        det_attempts=100,
        key_size=100
    )

    print("LOADED")
    
    do_RBE(ctx, f"{dataset}_Wid200_", N=200000, kmeans=False, top=False, random=False, inner_dim=200)
    
    del ctx
    gc.collect()




DATASET =  yugioh
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013
LOADED


/tmp/ipykernel_1002755/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

self.embed_users['train'].shape =  (2260, 100)
self.embed_games.shape =  (10031, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 10031)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (2260, 100)
dssm_dim = 100, inner_dim=200

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0487 tf.Tensor(37.382473, shape=(), dtype=float32) 100
slice = key, score = 0.0487
np.mean(results), mse, len(results) =  0.04720796460176992 tf.Tensor(35.60154, shape=(), dtype=float32) 2260
slice = train, score = 0.04720796460176992
np.mean(results), mse, len(results) =  0.04831194471865746 tf.Tensor(36.1424, shape=(), dtype=float32) 1013
slice = test, score = 0.04831194471865746
np.mean(results), mse, len(results) =  0.04720796460176992 tf.Tensor(35.60154, shape=(), dtype=float32) 2260
np.mean(results), mse, len(results) =  0.04831194471865746 tf.Tensor(36.1424, shape=(), dtype=float32) 1013
Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.0472_0.

np.mean(results), mse, len(results) =  0.5693385982230997 tf.Tensor(862.9391, shape=(), dtype=float32) 1013
Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.6193_0.5693.npz ...

=== Iteration 11000 ===
np.mean(results), mse, len(results) =  0.6091000000000001 tf.Tensor(974.0581, shape=(), dtype=float32) 100
slice = key, score = 0.6091000000000001
np.mean(results), mse, len(results) =  0.6246150442477877 tf.Tensor(964.3178, shape=(), dtype=float32) 2260
slice = train, score = 0.6246150442477877
np.mean(results), mse, len(results) =  0.573751233958539 tf.Tensor(970.6004, shape=(), dtype=float32) 1013
slice = test, score = 0.573751233958539
np.mean(results), mse, len(results) =  0.6246150442477877 tf.Tensor(964.3178, shape=(), dtype=float32) 2260
np.mean(results), mse, len(results) =  0.573751233958539 tf.Tensor(970.6004, shape=(), dtype=float32) 1013
Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.6246_0.5738.npz ...

=== Iteration 12000 ===
np.mean(results), mse, len(results) =  0.6098 tf.Tensor(1

...


=== Iteration 122000 ===
np.mean(results), mse, len(results) =  0.2975000000000001 tf.Tensor(117367.6, shape=(), dtype=float32) 100
slice = key, score = 0.2975000000000001
np.mean(results), mse, len(results) =  0.38482583913869545 tf.Tensor(118180.086, shape=(), dtype=float32) 1579
slice = train, score = 0.38482583913869545
np.mean(results), mse, len(results) =  0.3227083333333334 tf.Tensor(118101.8, shape=(), dtype=float32) 720
slice = test, score = 0.3227083333333334

=== Iteration 123000 ===
np.mean(results), mse, len(results) =  0.30010000000000003 tf.Tensor(118678.36, shape=(), dtype=float32) 100
slice = key, score = 0.30010000000000003
np.mean(results), mse, len(results) =  0.38628879037365427 tf.Tensor(119746.98, shape=(), dtype=float32) 1579
slice = train, score = 0.38628879037365427
np.mean(results), mse, len(results) =  0.3257777777777778 tf.Tensor(119664.74, shape=(), dtype=float32) 720
slice = test, score = 0.3257777777777778

=== Iteration 124000 ===
np.mean(results), mse

np.mean(results), mse, len(results) =  0.3919189360354655 tf.Tensor(149829.7, shape=(), dtype=float32) 1579
slice = train, score = 0.3919189360354655
np.mean(results), mse, len(results) =  0.32969444444444446 tf.Tensor(149634.16, shape=(), dtype=float32) 720
slice = test, score = 0.32969444444444446

=== Iteration 140000 ===
np.mean(results), mse, len(results) =  0.3039 tf.Tensor(150257.6, shape=(), dtype=float32) 100
slice = key, score = 0.3039
np.mean(results), mse, len(results) =  0.38986700443318556 tf.Tensor(151670.38, shape=(), dtype=float32) 1579
slice = train, score = 0.38986700443318556
np.mean(results), mse, len(results) =  0.32618055555555553 tf.Tensor(151537.69, shape=(), dtype=float32) 720
slice = test, score = 0.32618055555555553

=== Iteration 141000 ===
np.mean(results), mse, len(results) =  0.3043 tf.Tensor(152539.81, shape=(), dtype=float32) 100
slice = key, score = 0.3043
np.mean(results), mse, len(results) =  0.3920329322355922 tf.Tensor(153802.86, shape=(), dtype=f

np.mean(results), mse, len(results) =  0.32755555555555554 tf.Tensor(182333.44, shape=(), dtype=float32) 720
slice = test, score = 0.32755555555555554

=== Iteration 156000 ===
np.mean(results), mse, len(results) =  0.31200000000000006 tf.Tensor(182916.81, shape=(), dtype=float32) 100
slice = key, score = 0.31200000000000006
np.mean(results), mse, len(results) =  0.3933312222925902 tf.Tensor(184189.3, shape=(), dtype=float32) 1579
slice = train, score = 0.3933312222925902
np.mean(results), mse, len(results) =  0.3302083333333334 tf.Tensor(184149.08, shape=(), dtype=float32) 720
slice = test, score = 0.3302083333333334

=== Iteration 157000 ===
np.mean(results), mse, len(results) =  0.306 tf.Tensor(184295.7, shape=(), dtype=float32) 100
slice = key, score = 0.306
np.mean(results), mse, len(results) =  0.3913046231792274 tf.Tensor(186617.47, shape=(), dtype=float32) 1579
slice = train, score = 0.3913046231792274
np.mean(results), mse, len(results) =  0.3270972222222222 tf.Tensor(186265.0

np.mean(results), mse, len(results) =  0.326625 tf.Tensor(220821.48, shape=(), dtype=float32) 720
slice = test, score = 0.326625

=== Iteration 174000 ===
np.mean(results), mse, len(results) =  0.3065 tf.Tensor(219908.17, shape=(), dtype=float32) 100
slice = key, score = 0.3065
np.mean(results), mse, len(results) =  0.39189993666877765 tf.Tensor(222641.84, shape=(), dtype=float32) 1579
slice = train, score = 0.39189993666877765
np.mean(results), mse, len(results) =  0.3274722222222222 tf.Tensor(222251.03, shape=(), dtype=float32) 720
slice = test, score = 0.3274722222222222

=== Iteration 175000 ===
np.mean(results), mse, len(results) =  0.3069 tf.Tensor(222777.2, shape=(), dtype=float32) 100
slice = key, score = 0.3069
np.mean(results), mse, len(results) =  0.39102596580114 tf.Tensor(225106.2, shape=(), dtype=float32) 1579
slice = train, score = 0.39102596580114
np.mean(results), mse, len(results) =  0.3266527777777778 tf.Tensor(224684.77, shape=(), dtype=float32) 720
slice = test, sc

np.mean(results), mse, len(results) =  0.3275138888888889 tf.Tensor(256025.5, shape=(), dtype=float32) 720
slice = test, score = 0.3275138888888889

=== Iteration 190000 ===
np.mean(results), mse, len(results) =  0.3097 tf.Tensor(255783.45, shape=(), dtype=float32) 100
slice = key, score = 0.3097
np.mean(results), mse, len(results) =  0.39252691576947435 tf.Tensor(259219.5, shape=(), dtype=float32) 1579
slice = train, score = 0.39252691576947435
np.mean(results), mse, len(results) =  0.32729166666666665 tf.Tensor(258656.75, shape=(), dtype=float32) 720
slice = test, score = 0.32729166666666665

=== Iteration 191000 ===
np.mean(results), mse, len(results) =  0.3111 tf.Tensor(258022.88, shape=(), dtype=float32) 100
slice = key, score = 0.3111
np.mean(results), mse, len(results) =  0.39471817606079795 tf.Tensor(261766.19, shape=(), dtype=float32) 1579
slice = train, score = 0.39471817606079795
np.mean(results), mse, len(results) =  0.3294166666666667 tf.Tensor(261023.7, shape=(), dtype=fl

 'Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.6772_0.5879.npz ...',
 'Saving ./GE_QE_RBExCoItem_pro_wrestling_Wid200__0.6759_0.5264.npz .
 'Saving ./GE_QE_RBExCoItem_star_trek_Wid200__0.4404_0.3886.npz ...
 'Saving ./GE_QE_RBExCoItem_doctor_who_Wid200__0.3467_0.3006.npz ...
 'Saving ./GE_QE_RBExCoItem_military_Wid200__0.3917_0.329.npz ... IP

In [16]:
[x for x in s.split("\n") if x.startswith("Saving")]

['Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.0472_0.0483.npz ...',
 'Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.5713_0.5392.npz ...',
 'Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.5888_0.5523.npz ...',
 'Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.5988_0.5604.npz ...',
 'Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.6044_0.5638.npz ...',
 'Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.6095_0.5658.npz ...',
 'Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.6103_0.5661.npz ...',
 'Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.6162_0.5705.npz ...',
 'Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.6163_0.5692.npz ...',
 'Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.6184_0.5714.npz ...',
 'Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.6193_0.5693.npz ...',
 'Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.6246_0.5738.npz ...',
 'Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.6278_0.5747.npz ...',
 'Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.6291_0.5746.npz ...',
 'Saving ./GE_QE_RBExCoItem_yugioh_Wid200__0.6332_0.5764.npz .

In [ ]:
+InnerDim + E + H

In [17]:
R = np.load('./R9650test.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=False
)

Best det =  0.0
Current de =  0.0
100 6654 2895


In [18]:
do(ctx, f"MsMarco_")

/tmp/ipykernel_1002755/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

np.mean(results), mse, len(results) =  0.6243763149984972 0.0010132224 6654
np.mean(results), mse, len(results) =  0.6145181347150259 0.0010545495 2895
./GE_QE_AnnCURxCoItem_MsMarco__6145.npz


/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


np.mean(results), mse, len(results) =  0.5727908025247971 0.0012930894 6654
np.mean(results), mse, len(results) =  0.563419689119171 0.0013292843 2895
./GE_QE_AnnCURxKMeans_MsMarco__5634.npz
np.mean(results), mse, len(results) =  0.5655966336038474 0.0017333137 6654
np.mean(results), mse, len(results) =  0.553972366148532 0.0017844876 2895
./GE_QE_AnnCURxTop_MsMarco__554.npz
np.mean(results), mse, len(results) =  0.5874256086564472 0.0013426415 6654
np.mean(results), mse, len(results) =  0.5773436960276339 0.0013754577 2895
./GE_QE_AnnCURxRandom_MsMarco__5773.npz


In [ ]:
do_RBE(ctx, f"MsMarco_", N=300000, kmeans=False, top=False, random=False)

/tmp/ipykernel_1002755/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

self.embed_users['train'].shape =  (6654, 100)
self.embed_games.shape =  (77897, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 77897)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (6654, 100)
dssm_dim = 100, inner_dim=100

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0012 tf.Tensor(39.215702, shape=(), dtype=float32) 100
slice = key, score = 0.0012
np.mean(results), mse, len(results) =  0.001248872858431019 tf.Tensor(38.206448, shape=(), dtype=float32) 6654
slice = train, score = 0.001248872858431019
np.mean(results), mse, len(results) =  0.0012918825561312607 tf.Tensor(38.22677, shape=(), dtype=float32) 2895
slice = test, score = 0.0012918825561312607
np.mean(results), mse, len(results) =  0.001248872858431019 tf.Tensor(38.206448, shape=(), dtype=float32) 6654
np.mean(results), mse, len(results) =  0.0012918825561312607 tf.Tensor(38.22677, shape=(), dtype=float32) 2895
Saving ./GE_QE_RBExCoItem_MsMarco__0.

np.mean(results), mse, len(results) =  0.6533047790802525 tf.Tensor(341.22498, shape=(), dtype=float32) 6654
slice = train, score = 0.6533047790802525
np.mean(results), mse, len(results) =  0.6368566493955096 tf.Tensor(343.85748, shape=(), dtype=float32) 2895
slice = test, score = 0.6368566493955096
np.mean(results), mse, len(results) =  0.6533047790802525 tf.Tensor(341.22498, shape=(), dtype=float32) 6654
np.mean(results), mse, len(results) =  0.6368566493955096 tf.Tensor(343.85748, shape=(), dtype=float32) 2895
Saving ./GE_QE_RBExCoItem_MsMarco__0.6533_0.6369.npz ...

=== Iteration 12000 ===
np.mean(results), mse, len(results) =  0.6232 tf.Tensor(370.6384, shape=(), dtype=float32) 100
slice = key, score = 0.6232
np.mean(results), mse, len(results) =  0.6542921550946799 tf.Tensor(364.12335, shape=(), dtype=float32) 6654
slice = train, score = 0.6542921550946799
np.mean(results), mse, len(results) =  0.6376580310880829 tf.Tensor(367.85895, shape=(), dtype=float32) 2895
slice = test, sc

np.mean(results), mse, len(results) =  0.642545768566494 tf.Tensor(560.9572, shape=(), dtype=float32) 2895
slice = test, score = 0.642545768566494
np.mean(results), mse, len(results) =  0.6612368500150285 tf.Tensor(556.45013, shape=(), dtype=float32) 6654
np.mean(results), mse, len(results) =  0.642545768566494 tf.Tensor(560.9572, shape=(), dtype=float32) 2895
Saving ./GE_QE_RBExCoItem_MsMarco__0.6612_0.6425.npz ...

=== Iteration 23000 ===
np.mean(results), mse, len(results) =  0.634 tf.Tensor(563.64716, shape=(), dtype=float32) 100
slice = key, score = 0.634
np.mean(results), mse, len(results) =  0.6607318905921251 tf.Tensor(559.1675, shape=(), dtype=float32) 6654
slice = train, score = 0.6607318905921251
np.mean(results), mse, len(results) =  0.6416891191709845 tf.Tensor(564.5218, shape=(), dtype=float32) 2895
slice = test, score = 0.6416891191709845

=== Iteration 24000 ===
np.mean(results), mse, len(results) =  0.6347 tf.Tensor(577.3249, shape=(), dtype=float32) 100
slice = key, s

...

np.mean(results), mse, len(results) =  0.6503350604490502 tf.Tensor(717.7144, shape=(), dtype=float32) 2895
slice = test, score = 0.6503350604490502

=== Iteration 75000 ===
np.mean(results), mse, len(results) =  0.6439 tf.Tensor(728.13635, shape=(), dtype=float32) 100
slice = key, score = 0.6439
np.mean(results), mse, len(results) =  0.6732536819957919 tf.Tensor(712.7338, shape=(), dtype=float32) 6654
slice = train, score = 0.6732536819957919
np.mean(results), mse, len(results) =  0.6505423143350604 tf.Tensor(720.00995, shape=(), dtype=float32) 2895
slice = test, score = 0.6505423143350604

=== Iteration 76000 ===
np.mean(results), mse, len(results) =  0.6429 tf.Tensor(732.65576, shape=(), dtype=float32) 100
slice = key, score = 0.6429
np.mean(results), mse, len(results) =  0.6731875563570785 tf.Tensor(714.3376, shape=(), dtype=float32) 6654
slice = train, score = 0.6731875563570785
np.mean(results), mse, len(results) =  0.6501623488773747 tf.Tensor(721.9832, shape=(), dtype=float32) 

np.mean(results), mse, len(results) =  0.6506666666666667 tf.Tensor(776.46954, shape=(), dtype=float32) 2895
slice = test, score = 0.6506666666666667
np.mean(results), mse, len(results) =  0.6749624286143674 tf.Tensor(767.88776, shape=(), dtype=float32) 6654
np.mean(results), mse, len(results) =  0.6506666666666667 tf.Tensor(776.46954, shape=(), dtype=float32) 2895
Saving ./GE_QE_RBExCoItem_MsMarco__0.675_0.6507.npz ...

=== Iteration 90000 ===
np.mean(results), mse, len(results) =  0.6439 tf.Tensor(795.7069, shape=(), dtype=float32) 100
slice = key, score = 0.6439
np.mean(results), mse, len(results) =  0.6749834685903217 tf.Tensor(779.8352, shape=(), dtype=float32) 6654
slice = train, score = 0.6749834685903217
np.mean(results), mse, len(results) =  0.6507357512953368 tf.Tensor(787.3994, shape=(), dtype=float32) 2895
slice = test, score = 0.6507357512953368
np.mean(results), mse, len(results) =  0.6749834685903217 tf.Tensor(779.8352, shape=(), dtype=float32) 6654
np.mean(results), mse

np.mean(results), mse, len(results) =  0.6517685664939551 tf.Tensor(848.1127, shape=(), dtype=float32) 2895
slice = test, score = 0.6517685664939551

=== Iteration 103000 ===
np.mean(results), mse, len(results) =  0.6459 tf.Tensor(873.34814, shape=(), dtype=float32) 100
slice = key, score = 0.6459
np.mean(results), mse, len(results) =  0.6757754733994589 tf.Tensor(856.3915, shape=(), dtype=float32) 6654
slice = train, score = 0.6757754733994589
np.mean(results), mse, len(results) =  0.651651122625216 tf.Tensor(865.867, shape=(), dtype=float32) 2895
slice = test, score = 0.651651122625216

=== Iteration 104000 ===
np.mean(results), mse, len(results) =  0.6436 tf.Tensor(865.0131, shape=(), dtype=float32) 100
slice = key, score = 0.6436
np.mean(results), mse, len(results) =  0.6763931469792605 tf.Tensor(849.8991, shape=(), dtype=float32) 6654
slice = train, score = 0.6763931469792605
np.mean(results), mse, len(results) =  0.6520449050086357 tf.Tensor(857.5753, shape=(), dtype=float32) 289


=== Iteration 116000 ===
np.mean(results), mse, len(results) =  0.6442 tf.Tensor(951.57996, shape=(), dtype=float32) 100
slice = key, score = 0.6442
np.mean(results), mse, len(results) =  0.6778238653441538 tf.Tensor(940.08246, shape=(), dtype=float32) 6654
slice = train, score = 0.6778238653441538
np.mean(results), mse, len(results) =  0.6530811744386874 tf.Tensor(948.6093, shape=(), dtype=float32) 2895
slice = test, score = 0.6530811744386874

=== Iteration 117000 ===
np.mean(results), mse, len(results) =  0.6449000000000001 tf.Tensor(966.88196, shape=(), dtype=float32) 100
slice = key, score = 0.6449000000000001
np.mean(results), mse, len(results) =  0.6780087165614667 tf.Tensor(954.7813, shape=(), dtype=float32) 6654
slice = train, score = 0.6780087165614667
np.mean(results), mse, len(results) =  0.6529395509499136 tf.Tensor(962.8087, shape=(), dtype=float32) 2895
slice = test, score = 0.6529395509499136

=== Iteration 118000 ===
np.mean(results), mse, len(results) =  0.647 tf.Ten

np.mean(results), mse, len(results) =  0.6785362188157499 tf.Tensor(1088.1749, shape=(), dtype=float32) 6654
slice = train, score = 0.6785362188157499
np.mean(results), mse, len(results) =  0.6532746113989637 tf.Tensor(1097.1973, shape=(), dtype=float32) 2895
slice = test, score = 0.6532746113989637

=== Iteration 134000 ===
np.mean(results), mse, len(results) =  0.6468 tf.Tensor(1080.7726, shape=(), dtype=float32) 100
slice = key, score = 0.6468
np.mean(results), mse, len(results) =  0.6787165614667869 tf.Tensor(1080.0397, shape=(), dtype=float32) 6654
slice = train, score = 0.6787165614667869
np.mean(results), mse, len(results) =  0.6530017271157168 tf.Tensor(1087.3562, shape=(), dtype=float32) 2895
slice = test, score = 0.6530017271157168

=== Iteration 135000 ===
np.mean(results), mse, len(results) =  0.6448 tf.Tensor(1111.011, shape=(), dtype=float32) 100
slice = key, score = 0.6448
np.mean(results), mse, len(results) =  0.6780958821761346 tf.Tensor(1108.8506, shape=(), dtype=floa